# Statistical Modeling & Risk-Based Pricing

## Objective
Build and evaluate predictive models that form the core of a dynamic, risk-based pricing system.

## Modeling Goals
1. **Claim Severity Prediction**: Predict TotalClaims for policies with claims
   - Target: TotalClaims (subset where claims > 0)
   - Evaluation: RMSE and R²

2. **Claim Probability Prediction**: Predict probability of a claim
   - Target: Binary (claim occurred or not)
   - Evaluation: Accuracy, Precision, Recall, F1, ROC-AUC

3. **Risk-Based Premium Calculation**:
   - Premium = (P(claim) × Predicted Severity) + Expense Loading + Profit Margin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import DataLoader
from src.modeling import (
    InsuranceModelingPipeline,
    engineer_features,
    calculate_risk_based_premium,
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

## 1. Load and Prepare Data

In [ ]:
# Load data
data_path = '../data/insurance_data.csv'
loader = DataLoader(data_path)

try:
    df = loader.load_data()
    print(f'Data loaded successfully: {df.shape}')
    
    # Create derived metrics
    df = loader.create_derived_metrics(df)
    print('Derived metrics created')
    
    # Engineer features
    df = engineer_features(df)
    print('Features engineered')
    
except FileNotFoundError:
    print(f'Data file not found at {data_path}')
    df = None

## 2. Data Preparation for Modeling

In [ ]:
if df is not None:
    # Handle missing values
    df_clean = loader.handle_missing_values(strategy='impute', threshold=0.5)
    print(f'Data after handling missing values: {df_clean.shape}')
    
    # Select features for modeling
    # Exclude target variables and identifiers
    exclude_cols = ['TotalClaims', 'TotalPremium', 'LossRatio', 'Margin', 'ClaimIndicator']
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    print(f'\nFeatures selected for modeling: {len(feature_cols)}')
    print(feature_cols[:10], '...')

## 3. Model 1: Claim Severity Prediction (Regression)

In [ ]:
if df is not None:
    print('\nMODEL 1: CLAIM SEVERITY PREDICTION')
    print('='*60)
    print('Target: TotalClaims (for policies with claims > 0)')
    print('Evaluation: RMSE, R², MAE')
    
    # Filter to policies with claims
    df_claims = df_clean[df_clean['TotalClaims'] > 0].copy()
    print(f'\nPolicies with claims: {len(df_claims)} ({len(df_claims)/len(df_clean)*100:.1f}%)')
    
    # Prepare features and target
    X_severity = df_claims[feature_cols].copy()
    y_severity = df_claims['TotalClaims'].copy()
    
    # Remove rows with NaN
    valid_idx = ~(X_severity.isna().any(axis=1) | y_severity.isna())
    X_severity = X_severity[valid_idx]
    y_severity = y_severity[valid_idx]
    
    print(f'Valid samples for modeling: {len(X_severity)}')
    
    # Split data
    X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
        X_severity, y_severity, test_size=0.2, random_state=42
    )
    
    print(f'Train set: {len(X_train_sev)}, Test set: {len(X_test_sev)}')

In [ ]:
if df is not None and len(X_severity) > 0:
    # Initialize pipeline
    pipeline = InsuranceModelingPipeline(random_state=42)
    
    # Preprocess features
    X_train_sev_processed = pipeline.preprocess_features(X_train_sev, fit=True)
    X_test_sev_processed = pipeline.preprocess_features(X_test_sev, fit=False)
    
    print('Features preprocessed')
    
    # Train models
    print('\nTraining models...')
    
    metrics_lr = pipeline.train_linear_regression(
        X_train_sev_processed, y_train_sev, X_test_sev_processed, y_test_sev
    )
    print('✓ Linear Regression trained')
    
    metrics_rf = pipeline.train_random_forest_regressor(
        X_train_sev_processed, y_train_sev, X_test_sev_processed, y_test_sev,
        n_estimators=100, max_depth=10
    )
    print('✓ Random Forest trained')
    
    try:
        metrics_xgb = pipeline.train_xgboost_regressor(
            X_train_sev_processed, y_train_sev, X_test_sev_processed, y_test_sev,
            n_estimators=100, max_depth=5, learning_rate=0.1
        )
        print('✓ XGBoost trained')
    except ImportError:
        print('⚠ XGBoost not available')
        metrics_xgb = None

In [ ]:
if df is not None and len(X_severity) > 0:
    # Model comparison
    models_to_compare = [metrics_lr, metrics_rf]
    if metrics_xgb:
        models_to_compare.append(metrics_xgb)
    
    comparison_df = pipeline.get_model_comparison(models_to_compare)
    print('\nCLAIM SEVERITY MODEL COMPARISON')
    print('='*80)
    print(comparison_df.to_string(index=False))
    
    # Best model
    best_model_idx = comparison_df['Test RMSE'].idxmin()
    best_model_name = comparison_df.loc[best_model_idx, 'Model']
    print(f'\nBest Model: {best_model_name}')

In [ ]:
if df is not None and len(X_severity) > 0:
    # Feature importance for best model
    if best_model_name == 'Random Forest Regressor':
        print('\nTop 10 Features - Random Forest (Claim Severity)')
        print('='*60)
        print(metrics_rf['feature_importance'].head(10).to_string(index=False))
    elif best_model_name == 'XGBoost Regressor':
        print('\nTop 10 Features - XGBoost (Claim Severity)')
        print('='*60)
        print(metrics_xgb['feature_importance'].head(10).to_string(index=False))

## 4. Model 2: Claim Probability Prediction (Classification)

In [ ]:
if df is not None:
    print('\nMODEL 2: CLAIM PROBABILITY PREDICTION')
    print('='*60)
    print('Target: Binary (claim occurred or not)')
    print('Evaluation: Accuracy, Precision, Recall, F1, ROC-AUC')
    
    # Prepare features and target
    X_prob = df_clean[feature_cols].copy()
    y_prob = df_clean['ClaimIndicator'].copy()
    
    # Remove rows with NaN
    valid_idx = ~(X_prob.isna().any(axis=1) | y_prob.isna())
    X_prob = X_prob[valid_idx]
    y_prob = y_prob[valid_idx]
    
    print(f'\nTotal samples: {len(X_prob)}')
    print(f'Claims (1): {(y_prob == 1).sum()} ({(y_prob == 1).sum()/len(y_prob)*100:.1f}%)')
    print(f'No Claims (0): {(y_prob == 0).sum()} ({(y_prob == 0).sum()/len(y_prob)*100:.1f}%)')
    
    # Split data
    X_train_prob, X_test_prob, y_train_prob, y_test_prob = train_test_split(
        X_prob, y_prob, test_size=0.2, random_state=42, stratify=y_prob
    )
    
    print(f'\nTrain set: {len(X_train_prob)}, Test set: {len(X_test_prob)}')

In [ ]:
if df is not None and len(X_prob) > 0:
    # Initialize pipeline
    pipeline_clf = InsuranceModelingPipeline(random_state=42)
    
    # Preprocess features
    X_train_prob_processed = pipeline_clf.preprocess_features(X_train_prob, fit=True)
    X_test_prob_processed = pipeline_clf.preprocess_features(X_test_prob, fit=False)
    
    print('Features preprocessed')
    
    # Train classifier
    print('\nTraining classifier...')
    
    metrics_clf = pipeline_clf.train_random_forest_classifier(
        X_train_prob_processed, y_train_prob, X_test_prob_processed, y_test_prob,
        n_estimators=100, max_depth=10
    )
    print('✓ Random Forest Classifier trained')
    
    # Display metrics
    print('\nCLAIM PROBABILITY MODEL PERFORMANCE')
    print('='*60)
    print(f'Test Accuracy: {metrics_clf["test_accuracy"]:.4f}')
    print(f'Test Precision: {metrics_clf["test_precision"]:.4f}')
    print(f'Test Recall: {metrics_clf["test_recall"]:.4f}')
    print(f'Test F1-Score: {metrics_clf["test_f1"]:.4f}')
    print(f'Test ROC-AUC: {metrics_clf["test_roc_auc"]:.4f}')

In [ ]:
if df is not None and len(X_prob) > 0:
    # Feature importance
    print('\nTop 10 Features - Claim Probability')
    print('='*60)
    print(metrics_clf['feature_importance'].head(10).to_string(index=False))

## 5. Risk-Based Premium Calculation

In [ ]:
if df is not None and len(X_prob) > 0 and len(X_severity) > 0:
    print('\nRISK-BASED PREMIUM CALCULATION')
    print('='*60)
    print('Formula: Premium = (P(claim) × Predicted Severity) + Expense Loading + Profit Margin')
    
    # Get predictions on test set
    y_prob_pred = pipeline_clf.models['rf_classifier'].predict_proba(X_test_prob_processed)[:, 1]
    y_severity_pred = pipeline.models['random_forest'].predict(X_test_sev_processed)
    
    # Calculate risk-based premiums
    # Use average predicted severity for simplicity
    avg_severity = y_severity_pred.mean()
    
    risk_based_premiums = []
    for prob in y_prob_pred[:100]:  # Sample of 100
        premium = calculate_risk_based_premium(
            claim_probability=prob,
            predicted_severity=avg_severity,
            expense_loading=0.15,
            profit_margin=0.20
        )
        risk_based_premiums.append(premium)
    
    print(f'\nAverage Predicted Severity: R{avg_severity:,.2f}')
    print(f'Average Risk-Based Premium: R{np.mean(risk_based_premiums):,.2f}')
    print(f'Min Premium: R{np.min(risk_based_premiums):,.2f}')
    print(f'Max Premium: R{np.max(risk_based_premiums):,.2f}')
    print(f'Std Dev: R{np.std(risk_based_premiums):,.2f}')

## 6. Model Interpretability with SHAP

In [ ]:
if df is not None and len(X_severity) > 0:
    print('\nMODEL INTERPRETABILITY - SHAP ANALYSIS')
    print('='*60)
    print('Attempting to generate SHAP values for feature importance...')
    
    try:
        # Get SHAP values for severity model
        shap_values = pipeline.get_shap_values('random_forest', X_test_sev_processed[:100])
        print('✓ SHAP values calculated successfully')
        print(f'SHAP values shape: {shap_values.shape}')
        print('\nNote: SHAP plots require interactive environment')
        print('Feature importance from SHAP:')
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        shap_importance = pd.DataFrame({
            'Feature': pipeline.feature_names,
            'Mean |SHAP|': mean_abs_shap
        }).sort_values('Mean |SHAP|', ascending=False)
        print(shap_importance.head(10).to_string(index=False))
    except ImportError:
        print('⚠ SHAP not available. Install with: pip install shap')
    except Exception as e:
        print(f'⚠ SHAP analysis failed: {str(e)}')

## 7. Business Insights & Recommendations

In [ ]:
print('\nBUSINESS INSIGHTS & RECOMMENDATIONS')
print('='*80)

if df is not None and len(X_severity) > 0:
    print('\n1. CLAIM SEVERITY MODELING')
    print(f'   Best Model: {best_model_name}')
    print(f'   Test RMSE: R{comparison_df.loc[best_model_idx, "Test RMSE"]:.2f}')
    print(f'   Test R²: {comparison_df.loc[best_model_idx, "Test R²"]:.4f}')
    print('   Interpretation: Model explains', f'{comparison_df.loc[best_model_idx, "Test R²"]*100:.1f}%', 'of variance in claim amounts')
    print('   Action: Use for estimating financial liability per policy')

if df is not None and len(X_prob) > 0:
    print('\n2. CLAIM PROBABILITY MODELING')
    print(f'   Model: Random Forest Classifier')
    print(f'   Test Accuracy: {metrics_clf["test_accuracy"]:.4f}')
    print(f'   Test ROC-AUC: {metrics_clf["test_roc_auc"]:.4f}')
    print('   Interpretation: Model predicts claim probability with', f'{metrics_clf["test_roc_auc"]*100:.1f}%', 'AUC')
    print('   Action: Use for risk stratification and premium calculation')

print('\n3. RISK-BASED PRICING FRAMEWORK')
print('   Formula: Premium = (P(claim) × Predicted Severity) + Expense Loading + Profit Margin')
print('   Benefits:')
print('   - Aligns premiums with actual risk')
print('   - Improves profitability for low-risk segments')
print('   - Enables competitive pricing for good risks')
print('   - Supports data-driven marketing strategy')

print('\n4. IMPLEMENTATION ROADMAP')
print('   Phase 1: Validate models on holdout data')
print('   Phase 2: A/B test new pricing on subset of customers')
print('   Phase 3: Monitor impact on loss ratio and customer acquisition')
print('   Phase 4: Full rollout with continuous monitoring')